In [1]:
from pathlib import Path
import pandas as pd

# ===============================
# CONFIGURACIÓN
# ===============================
CARPETA = Path(r"C:\Users\luisf\IQ Tech\DashboardRotacion\Stock Odoo Cuatitlan")
ARCHIVO = CARPETA / "rotacion_inventario_base_dashboard_odoo_autoazur.xlsx"
HOJA = "ventas_conjunto_detalle"

# ===============================
# CARGA
# ===============================
df = pd.read_excel(ARCHIVO, sheet_name=HOJA)
df.columns = [str(c).strip() for c in df.columns]

# ===============================
# VALIDACIONES
# ===============================
requeridas = ["fecha", "equipo_ventas", "venta_total"]
for col in requeridas:
    if col not in df.columns:
        raise KeyError(f"Columna obligatoria no encontrada: {col}")

# ===============================
# NORMALIZACIÓN
# ===============================
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["venta_total"] = pd.to_numeric(df["venta_total"], errors="coerce").fillna(0)

if "cantidad" in df.columns:
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0)
else:
    df["cantidad"] = 0

# ===============================
# LIMPIEZA DE CANAL
# ===============================
df["equipo_ventas"] = (
    df["equipo_ventas"]
    .astype(str)
    .str.strip()
    .replace(
        ["", "None", "nan", "NaN", "SIN ASIGNAR", "Sin asignar", "sin asignar"],
        pd.NA
    )
    .fillna("SIN CANAL")
)

# ===============================
# FILTRO ÚLTIMOS 3 MESES (90 días)
# ===============================
fecha_fin = df["fecha"].max().normalize()
fecha_inicio = fecha_fin - pd.Timedelta(days=89)

df_3m = df[
    df["fecha"].notna() &
    (df["fecha"] >= fecha_inicio) &
    (df["fecha"] <= fecha_fin)
].copy()

# ===============================
# AGRUPACIÓN POR PRODUCTO Y CANAL
# ===============================

resumen = (
    df_3m
    .groupby(["sku_madre", "equipo_ventas"], as_index=False)
    .agg(
        ventas_totales=("venta_total", "sum"),
        unidades_totales=("cantidad", "sum"),
    )
)

# ===============================
# TOTALES POR PRODUCTO
# ===============================

totales_producto = (
    resumen
    .groupby("sku_madre")
    .agg(
        total_ventas_producto=("ventas_totales", "sum"),
        total_unidades_producto=("unidades_totales", "sum"),
    )
    .reset_index()
)

# Unir los totales al resumen
resumen = resumen.merge(
    totales_producto,
    on="sku_madre",
    how="left"
)

# ===============================
# PORCENTAJES POR PRODUCTO
# ===============================

resumen["pct_ventas"] = (
    resumen["ventas_totales"]
    / resumen["total_ventas_producto"]
)

resumen["pct_unidades"] = (
    resumen["unidades_totales"]
    / resumen["total_unidades_producto"]
)

# Evitar NaN cuando el total sea 0
resumen["pct_ventas"] = resumen["pct_ventas"].fillna(0)
resumen["pct_unidades"] = resumen["pct_unidades"].fillna(0)

# ===============================
# ORDEN FINAL
# ===============================

resumen = (
    resumen
    .sort_values(
        ["sku_madre", "ventas_totales"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)
# ===============================
# ORDEN FINAL
# ===============================
resumen = (
    resumen
    .sort_values("ventas_totales", ascending=False)
    .reset_index(drop=True)
)

# ===============================
# RESULTADO
# ===============================
print(resumen.to_string(index=False))

# ===============================
# EXPORTAR
# ===============================
salida = CARPETA / "ventas_por_canal_ultimos_3_meses_con_porcentajes.xlsx"
with pd.ExcelWriter(salida, engine="openpyxl") as writer:
    resumen.to_excel(writer, sheet_name="ventas_por_canal_3m", index=False)

print(f"\nArchivo generado: {salida}")

sku_madre equipo_ventas  ventas_totales  unidades_totales  total_ventas_producto  total_unidades_producto  pct_ventas  pct_unidades
   IQ1463        Amazon      1298874.90               177             3289569.84                      295    0.394846      0.600000
   IQ1095       Walmart      1272061.49               769             1923983.78                     1171    0.661160      0.656704
   IQ1251        Amazon      1219778.33               999             1710537.06                     1361    0.713097      0.734019
   IQ1463        Coppel       992639.94                60             3289569.84                      295    0.301754      0.203390
   IQ1463 Mercado Libre       828552.00                48             3289569.84                      295    0.251872      0.162712
   IQ1501       Walmart       793125.61               101              803224.61                      102    0.987427      0.990196
   IQ1498 Mercado Libre       738126.30               135              99040